# 03 · The scene and its terrain

A digital twin is only as good as its geometry. This notebook opens the demo scene —
building footprints, a 70×70 terrain grid, and a tower list — and asks the geometric
questions that decide radio outcomes:

- is the link between the two towers actually line-of-sight?
- how much Fresnel clearance does it have, and at which frequency does that stop being true?
- what does putting the receiver at 1.5 m instead of "on the terrain" do to the answer?

**Which scene?** `load_scene()` takes the first of: an explicit path, `$ULAP_SCENE`,
your own `blender/scene_build/scene_manifest.json` if you have run the pipeline, and
otherwise the bundled sample built entirely from **open data** — OpenStreetMap
footprints and AWS terrain tiles (see `examples/data/fetch_open_scene.py`). The cell
below prints which one you got, and where every layer came from.

In [ ]:
import sys, pathlib
here = pathlib.Path.cwd()
EXAMPLES = next(p for p in [here, *here.parents] if (p / "ulap_demo").is_dir())
sys.path[:0] = [str(EXAMPLES), str(EXAMPLES.parent / "ulap-scope")]

import json
import numpy as np
import matplotlib.pyplot as plt
from ulap_demo import load_scene, PropagationModel, DATA_DIR
from ulap_demo.propagation import fresnel_radius_m, fresnel_nu, knife_edge_loss_db
from ulap_demo.plotting import (use_ulap_style, overlay_scene, plot_map, plot_terrain,
                                plot_profile, SUNGLOW, MARBLE_WHITE)

use_ulap_style()
scene = load_scene()
print(scene.summary())
print()
print(scene.sources())

## Open data is not free data

Look closely at the provenance above. OSM has excellent footprint *geometry* and almost
no building *heights* — so the sample assigns them from a per-type default table. The
pilot study used national LiDAR heights. That substitution is the single biggest
difference between this scene and the one behind `docs/renders/`, and you can see it in
the height histogram further down: a few discrete spikes where measurements would give
a continuous distribution.

Knowing that, and saying it out loud, is the difference between a twin and a guess.

## Validate the manifest with the pipeline's own checker

`ulap_scope.geo.validate_manifest` is what the pipeline runs before handing geometry to
Blender. Running it here is a good habit: a scene that fails this will fail later, more
expensively.

In [ ]:
from ulap_scope.geo import validate_manifest, REQUIRED_MANIFEST_KEYS

manifest = json.loads(scene.source_path.read_text())
problems = validate_manifest(manifest)
print("required keys:", ", ".join(REQUIRED_MANIFEST_KEYS))
print("problems   :", problems or "none — manifest is valid ✓")

If you are on the open-data sample you just saw it complain:

```
epsg should be 21292, got 32621
```

That is not a broken scene — it is a **real limitation of the validator**, which is
hard-coded to the pilot's national grid (EPSG:21292, Barbados 1938 / BWI). Our sample is
in UTM 21N, which is the metric CRS notebook 01 recommends for this longitude, and every
other check passes.

It is a genuinely good first contribution: make `validate_manifest` accept any projected
metric CRS and warn only on the known-bad ones (`NON_METRIC_EPSG` in `ulap_scope.study`
already lists them). Per [CONTRIBUTING.md](../../CONTRIBUTING.md), behaviour changes
start as an OpenSpec proposal.

In [ ]:
# What the manifest actually contains
print("EPSG      :", manifest["epsg"])
print("origin    :", [round(v, 2) for v in manifest["origin"]], "-> scene-local (0, 0)")
print("buildings :", len(manifest["buildings"]))
print("terrain   :", manifest["terrain"]["nx"], "x", manifest["terrain"]["ny"], "grid")

x0, y0 = scene.to_crs(0.0, 0.0)
print(f"\nscene origin in EPSG:{scene.epsg}: ({x0:.1f}, {y0:.1f})")
print("everything else in this notebook is in metres from that point.")

## The terrain

56 m of relief across a 2 km box is not flat, and it is not dramatic — which is exactly
the regime where people wrongly assume terrain does not matter.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_terrain(axes[0], scene, title="Terrain + footprints + towers")

hb = scene.building_heights()
axes[1].hist(hb, bins=30, color=SUNGLOW, alpha=0.85)
axes[1].axvline(hb.mean(), color=MARBLE_WHITE, ls="--", lw=1.2,
                label=f"mean {hb.mean():.1f} m")
axes[1].set_xlabel("building height [m]")
axes[1].set_ylabel("count")
axes[1].set_title(f"{len(hb)} buildings — low-rise, so terrain dominates")
axes[1].legend(); axes[1].grid(alpha=0.2)
fig.tight_layout()

relief = scene.terrain["zmax"] - scene.terrain["zmin"]
print(f"terrain relief {relief:.0f} m vs tallest building {hb.max():.0f} m "
      f"— a factor of {relief / hb.max():.0f}")
n_distinct = len(set(hb.round(2).tolist()))
print(f"{n_distinct} distinct building heights among {len(hb)} buildings"
      + ("  <-- heights are from a default table, not measured"
         if n_distinct < 20 else ""))

Two things in that histogram.

**Terrain dominates here.** The relief is an order of magnitude larger than the tallest
building, so in this study area hills shadow signal and buildings only nibble at it. In a
dense CBD the ranking flips — which is why the model lets you switch each term off and
measure its contribution (notebook 04 does exactly that).

**Count the distinct heights.** On the open-data sample there are only a handful, because
they came from a lookup table keyed on the OSM `building` tag. A LiDAR-derived scene gives
a smooth distribution instead. Neither is wrong; only one of them is a measurement, and a
study should record which it had — `scene.sources()` does.

## Rasterising the world

For fast propagation we need "how high is the world here?" as an array. `Scene.height_raster`
burns the footprints onto the terrain with a scanline polygon fill — no GDAL, no shapely.

In [ ]:
raster = scene.height_raster(cell_m=5.0)
bh = raster.building_height

print(f"raster {raster.top.shape[1]} x {raster.top.shape[0]} @ {raster.cell_m:g} m")
print(f"built-up pixels: {(bh > 0.1).sum():,} "
      f"({100 * (bh > 0.1).mean():.1f}% of the study area)")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_map(axes[0], raster.top, raster.extent, cmap="magma",
         title="Top-of-world surface (terrain + buildings)", cbar_label="height [m]")
plot_map(axes[1], np.where(bh > 0.1, bh, np.nan), raster.extent, cmap="plasma",
         title="Building extrusion above ground", cbar_label="height [m]")
fig.tight_layout()

## Is the tower-to-tower link line of sight?

Newton (30 m) to Rising Sun (24 m), 1.3 km apart across a valley. Backhaul planners live
and die by this question.

In [ ]:
newton = scene.tower("Newton")
rising = scene.tower("Rising Sun")

n_samples = 400
t = np.linspace(0, 1, n_samples)
px = newton.x + t * (rising.x - newton.x)
py = newton.y + t * (rising.y - newton.y)
D = float(np.hypot(rising.x - newton.x, rising.y - newton.y))
dist = t * D

ground = raster.sample_ground(px, py)
top = raster.sample_top(px, py)
z_a = newton.ground_z + newton.h
z_b = rising.ground_z + rising.h
ray = z_a + t * (z_b - z_a)

print(f"link length      : {D:.0f} m")
print(f"antenna heights  : Newton {z_a:.1f} m ASL, Rising Sun {z_b:.1f} m ASL")
print(f"highest obstacle : {top.max():.1f} m ASL")
print(f"minimum clearance: {(ray - top).min():.1f} m")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
plot_profile(ax, dist, ground, top, ray,
             title=f"Newton -> Rising Sun profile ({D:.0f} m)")

# first Fresnel zone at 3.5 GHz
d1, d2 = dist, D - dist
F1 = fresnel_radius_m(d1, d2, 3.5e9)
ax.fill_between(dist, ray - F1, ray + F1, color=SUNGLOW, alpha=0.16, zorder=3,
                label="1st Fresnel zone @ 3.5 GHz")
ax.plot(dist, ray - 0.6 * F1, color=SUNGLOW, ls=":", lw=1.0, alpha=0.8,
        label="60 % clearance line")
ax.legend(fontsize=8.5, loc="lower center", ncol=2)
fig.tight_layout()

### Clearance is frequency-dependent

The line of sight is geometric; **Fresnel clearance is not**. The first Fresnel zone is
fattest at low frequency, so an obstacle that a 28 GHz link ignores can obstruct a
700 MHz one. This backhaul hop is comfortable at every band — 20 m of clearance against a
12 m zone — which is what a good tower-to-tower link should look like:

In [ ]:
def clearance_table(d1, d2, ray, top, bands=(0.7, 1.8, 3.5, 10.0, 28.0)):
    print(f"{'band':>8} {'max F1':>9} {'clearance':>11} {'clearance/F1':>14} "
          f"{'60% rule':>10} {'nu':>7} {'diffraction':>13}")
    for ghz in bands:
        F = fresnel_radius_m(d1, d2, ghz * 1e9)
        clear = ray - top
        # Endpoints are excluded: F1 -> 0 there, and an obstacle sitting *at* an
        # antenna is not a knife edge (it is a siting problem).
        interior = slice(1, -1)
        ratio = float(np.min(clear[interior] / np.maximum(F[interior], 1e-9)))
        nu = float(np.max(fresnel_nu(-clear[interior], d1[interior], d2[interior],
                                     ghz * 1e9)))
        loss = float(knife_edge_loss_db(nu))
        print(f"{ghz:>6} GHz {F.max():>7.1f} m {clear.min():>9.1f} m {ratio:>13.2f} "
              f"{'pass' if ratio >= 0.6 else 'FAIL':>10} {nu:>7.2f} {loss:>10.1f} dB")

clearance_table(d1, d2, ray, top)

### Now a link that is *not* fine

Pick a receiver at handset height on the far side of the ridge from Newton and the same
table reads very differently. This is the regime most of your subscribers are in.

In [ ]:
# search the far side of the scene for the most obstructed handset position
Xs, Ys, _ = scene.grid(cell_m=40.0)
probe = PropagationModel(freq_hz=3.5e9, rx_height_m=1.5)
_, parts = probe.path_gain_db(scene, newton, Xs, Ys, return_parts=True)
blocked = np.where((parts["distance_m"] > 400) & (parts["distance_m"] < 1200),
                   parts["diffraction_db"], -np.inf)
iy, ix = np.unravel_index(np.argmax(blocked), blocked.shape)
rx_x, rx_y = float(Xs[iy, ix]), float(Ys[iy, ix])
print(f"worst shadowed handset within 400-1200 m: ({rx_x:.0f}, {rx_y:.0f}), "
      f"{parts['distance_m'][iy, ix]:.0f} m out, "
      f"{parts['diffraction_db'][iy, ix]:.1f} dB of diffraction loss @ 3.5 GHz")

In [ ]:
t2 = np.linspace(0, 1, 400)
qx = newton.x + t2 * (rx_x - newton.x)
qy = newton.y + t2 * (rx_y - newton.y)
D2 = float(np.hypot(rx_x - newton.x, rx_y - newton.y))
dist2 = t2 * D2
ground2 = raster.sample_ground(qx, qy)
top2 = raster.sample_top(qx, qy)
ray2 = (newton.ground_z + newton.h) + t2 * ((ground2[-1] + 1.5) - (newton.ground_z + newton.h))

fig, ax = plt.subplots(figsize=(11, 4.6))
plot_profile(ax, dist2, ground2, top2, ray2,
             title=f"Newton -> shadowed handset ({D2:.0f} m, RX at 1.5 m)")
F1b = fresnel_radius_m(dist2, D2 - dist2, 3.5e9)
ax.fill_between(dist2, ray2 - F1b, ray2 + F1b, color=SUNGLOW, alpha=0.16, zorder=3)
fig.tight_layout()

print()
clearance_table(dist2, D2 - dist2, ray2, top2)

The terrain cuts the ray, `nu` goes positive, and the diffraction penalty **grows with
frequency** — the same obstacle costs progressively more the shorter the wavelength. That
single mechanism is most of why the mmWave coverage map in the pilot study
(`docs/renders/mmwave_concrete_coverage.png`) is all holes.

## Receiver height is a planning decision, not a detail

The pilot study makes this point with two ray-traced maps: a **terrain-drape** coverage
map and a **ground-following 1.5 m** map. They look meaningfully different, and only one
of them answers "will a handset work here?".

We can show the same effect geometrically — how much of the scene each height can see
from a tower:

In [ ]:
X, Y, extent = scene.grid(cell_m=10.0)

# Fraction of the scene with an effectively clear first Fresnel zone.
def visible_fraction(tower, rx_height_m, freq_hz=3.5e9):
    m = PropagationModel(freq_hz=freq_hz, rx_height_m=rx_height_m)
    _, parts = m.path_gain_db(scene, tower, X, Y, return_parts=True)
    return parts["los"], float(parts["los"].mean())

los_15, f15 = visible_fraction(newton, 1.5)
los_10, f10 = visible_fraction(newton, 10.0)
los_30, f30 = visible_fraction(newton, 30.0)

print(f"clear-path fraction from Newton @ 3.5 GHz")
print(f"  handset height, 1.5 m : {100 * f15:.1f}%")
print(f"  rooftop CPE,   10.0 m : {100 * f10:.1f}%")
print(f"  drone,         30.0 m : {100 * f30:.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5.2))
for ax, (mask, h, frac) in zip(axes, [(los_15, 1.5, f15), (los_10, 10.0, f10),
                                      (los_30, 30.0, f30)]):
    ax.imshow(mask, origin="lower", extent=extent, cmap="cividis",
              interpolation="nearest", vmin=0, vmax=1)
    overlay_scene(ax, scene, label_towers=False, building_alpha=0.18)
    ax.set_title(f"RX at {h:g} m — {100 * frac:.0f}% clear path")
fig.suptitle("What Newton can 'see', by receiver height", y=1.0)
fig.tight_layout()

Every metre of receiver height buys coverage. This is why the study insists that planning
decisions be read off the **1.5 m ground-following** map: a terrain-draped map quietly
puts the receiver on the hilltops and flatters your network.

Below is the ray-traced pair from the pilot study, which shows the same story with full
multipath. Note these were produced from the pilot's **LiDAR** scene, not from the
open-data sample this notebook may have loaded — compare the shapes and the argument,
not pixel for pixel.

In [ ]:
from IPython.display import Image, display

renders = EXAMPLES.parent / "docs" / "renders"
for name, caption in [("coverage_pathgain_db_terrain.png", "ray-traced: terrain drape"),
                      ("coverage_pathgain_db_terrain_groundfollow.png",
                       "ray-traced: ground-following 1.5 m — use this one")]:
    p = renders / name
    if p.exists():
        print(caption)
        display(Image(filename=str(p), width=620))
    else:
        print("(missing render:", p, ")")

## Your turn

- Swap in your own scene: `load_scene("path/to/your/scene_manifest.json")` after running
  `ulap-scope preprocess` on your study.
- Change the profile endpoints to check a backhaul hop you care about.
- Drop `raster_cell_m` to 2 m and see whether the clear-path fractions move — that is a
  cheap way to find out whether your raster resolution is lying to you.

Next: **[04 · Coverage & SINR](04_coverage_and_sinr.ipynb)** — whole-area maps, band
sweeps, and what happens when you add a third tower.